# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR⁲ dataset (mass spectrometry protein profiles from human mast cells) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library (if not already installed)
!pip install mlcroissant

## 1. Data Loading
We'll load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Let's list all record sets and their fields. We'll reference all entities by their `@id` as required by Croissant and MLSchema.

Record sets and fields (@id, name):

In [ ]:
# List all record sets by @id
record_sets = dataset.metadata.record_set
print(f"Found {len(record_sets)} record sets:\n")

for rset in record_sets:
    print(f"- [@id] {rset['@id']}")
    print(f"  Name: {rset.get('name','[No name]')}")
    print("  Fields:")
    for field in rset.get('field', []):
        # Some fields may be dicts, some may be string @id
        if isinstance(field, dict):
            print(f"     - [@id] {field['@id']} (name: {field.get('name','')})")
        else:
            print(f"     - [@id] {field}")
    print()

## 3. Data Extraction
Extract the main record set and convert it to a pandas DataFrame.

**Note:** By convention, most protein mass spec Croissant datasets provide a main record set with all protein-level annotations (abundance, accession, description, etc). We'll list their IDs, then load one as an example.

Replace `<main_record_set_id>` and `<some_field_id>` below as appropriate!

In [ ]:
# Choose a record set to extract (update @id as needed!)
# For demonstration, let's find the first non-empty record set.
main_record_set_id = None
for rset in dataset.metadata.record_set:
    if rset.get('field'):
        main_record_set_id = rset['@id']
        break
print(f"Using record set: {main_record_set_id}\n")

# List all record set @id for future reference
all_record_set_ids = [rs['@id'] for rs in dataset.metadata.record_set]

# Extract records from all sets and load into DataFrames
dataframes = {}
for rs_id in all_record_set_ids:
    print(f"Loading data for record set {rs_id}")
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df

# Show columns of the main table
print(f"Fields in main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set. We'll select a numeric field (e.g. abundance, coverage, or MW), filter and normalize it, and group by another field (@id-based references are used throughout).

*Update `<numeric_field_id>` and `<group_field_id>` with correct field @id values as seen in the field list above!*

In [ ]:
# Example: use 'cr:field:mw' as a field if MW (molecular weight) is present.
# Adjust field IDs according to available IDs from section 2 output.
selected_df = dataframes[main_record_set_id]
numeric_field_id = None
# Try to auto-select a numeric field ('mw' or coverage or abundance)
for f in selected_df.columns:
    if 'mw' in f.lower() or 'abundance' in f.lower() or 'coverage' in f.lower():
        numeric_field_id = f
        break
if numeric_field_id is None:
    numeric_field_id = selected_df.columns[0]  # fallback to first

threshold = selected_df[numeric_field_id].quantile(0.90)  # Filter top 10% as example
filtered_df = selected_df[selected_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head(5))

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records (top 5):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a string/category field
group_field_id = None
for f in selected_df.columns:
    if 'description' in f.lower() or 'accession' in f.lower():
        group_field_id = f
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and any relationships. Requires matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution
plt.figure(figsize=(8,5))
sns.histplot(selected_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id exists and is categorical with low cardinality, plot boxplot
if group_field_id:
    value_counts = selected_df[group_field_id].value_counts()
    if value_counts.size < 20:
        plt.figure(figsize=(10,5))
        sns.boxplot(
            x=group_field_id,
            y=numeric_field_id,
            data=selected_df
        )
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- We loaded the dataset with `mlcroissant`, explored record sets and fields, and performed extraction and analysis referencing all entities via their `@id` fields.
- We filtered and visualized the main numeric field (e.g. protein molecular weight or abundance).
- This approach ensures transparent and reproducible FAIR dataset access.